In [1]:
def input(filename: str):
    with open(filename, 'r') as input:
        data = [x.strip() for x in input]

    return data

from typing import List, Optional
from dataclasses import dataclass
import re
import pprint
import math
import random
from collections import defaultdict, deque
import datetime

import numpy as np
from hypothesis import given, assume, strategies as st

In [51]:
dance = [
    's1',
    'x4/3',
    'pe/b',
    'pc/b',
    'pa/c'
]

In [67]:
# line = list('abcde')
line = list('abcdefghijklmnop')

In [57]:
dance = input('day16.txt')[0].split(',')

In [58]:
moves = []
for move in dance:
    move_dict = {}
    if move.startswith('s'):
        move_dict['type'] = 's'
        move_dict['value'] = int(move.removeprefix('s'))
    elif move.startswith('x'):
        move_dict['type'] = 'x'
        move_dict['value'] = list(map(int, move.removeprefix('x').split('/')))
    elif move.startswith('p'):
        move_dict['type'] = 'p'
        move_dict['value'] = move.removeprefix('p').split('/')
    moves.append(move_dict)

In [78]:
def catch_a_groove1(line):
    for move in dance:
        if move.startswith('s'):
            l = int(move.removeprefix('s'))
            line = line[-l:] + line[:-l]
        elif move.startswith('x'):
            ia, ib = list(map(int, move.removeprefix('x').split('/')))
            a = line[ia]
            b = line[ib]
            line[ia] = b
            line[ib] = a
        elif move.startswith('p'):
            a, b = move.removeprefix('p').split('/')
            ia = line.index(a)
            ib = line.index(b)
            line[ia] = b
            line[ib] = a

    return line

In [65]:
def catch_a_groove3(line):
    # fwd = {x: x for x in 'abcde'}
    # bck = {x: x for x in 'abcde'}
    fwd = {x: x for x in 'abcdefghijklmnop'}
    bck = {x: x for x in 'abcdefghijklmnop'}
    for move in moves:
        if move['type'] == 's':
            l = move['value']
            line = line[-l:] + line[:-l]
        elif move['type'] == 'x':
            ia, ib = move['value']
            a = line[ia]
            b = line[ib]
            line[ia] = b
            line[ib] = a
        elif move['type'] == 'p':
            # fwd = {a: a, b: b, c: c}, bck = {a: a, b: b, c: c}
            a, b = move['value']
            vala = fwd[a]  # vala = a
            valb = fwd[b]  # valb = b
            # after this, fwd = {a: b, b: a, c: c}
            fwd[b] = vala
            fwd[a] = valb
            # after this, bck = {a: b, b: a, c: c}
            # for k, v in fwd.items():
            #     bck[v] = k
            bck[vala] = b
            bck[valb] = a
    #         print(fwd, bck)
    # print(line)
    result = []
    for char in line:
        result.append(bck[char])
    return result

In [68]:
    # start with abcde
    # 'pe/b', => aecdb
    # 'pc/b', => aebdc
    # 'pa/c'  => cebda

catch_a_groove3(line)

['d',
 'o',
 'e',
 'a',
 'i',
 'm',
 'l',
 'b',
 'n',
 'p',
 'j',
 'c',
 'h',
 'f',
 'k',
 'g']

In [252]:
line2 = ''
for i in range(16):
    line2 += '{0:04b}'.format(i)

In [254]:
int(line2, 2)

81985529216486895

In [259]:
line2 = 0b0000000100100011010001010110011110001001101010111100110111101111
mask = 0xffffffffffffffff
tl = 64
byte_width = 4
ref = {y: i for i, y in enumerate(list('abcdefghijklmnop'))}

In [265]:
line3 = 0b00000001001000110100
mask = 0b11111111111111111111
tl = 20

In [291]:
def switch_by_index(line, ia, ib):
    maska = 0b1111 << (tl - (ia + 1) * 4)
    maskb = 0b1111 << (tl - (ib + 1) * 4)
    if ia < ib:
        a = line >> (abs(ib - ia) * 4) & maskb
        b = line << (abs(ia - ib) * 4) & maska
    else:
        a = line << (abs(ib - ia) * 4) & maskb
        b = line >> (abs(ia - ib) * 4) & maska
    
    return line & ~(maska | maskb) | (a | b)

In [239]:
def catch_a_groove2(line):
    for move in dance:
        if move.startswith('s'):
            l = int(move.removeprefix('s')) * byte_width
            line = ((line << (tl - l)) ^ (line >> l)) & mask
        if move.startswith('x'):
            ia, ib = list(map(int, move.removeprefix('x').split('/')))
            maska = 0b1111 << (tl - (ia + 1) * byte_width)
            maskb = 0b1111 << (tl - (ib + 1) * byte_width)
            if ia < ib:
                a = line >> (abs(ib - ia) * byte_width) & maskb
                b = line << (abs(ia - ib) * byte_width) & maska
            else:
                a = line << (abs(ib - ia) * byte_width) & maskb
                b = line >> (abs(ia - ib) * byte_width) & maska
            
            line = line & ~(maska | maskb) | (a | b)
        if move.startswith('p'):
            a, b = move.removeprefix('p').split('/')
            loca = ref[a]
            locb = ref[b]
            bytea = f'{loca:04b}'
            byteb = f'{locb:04b}'
            sline = f'{line:064b}'
            ia = -1
            ib = -1
            for i in range(0, tl, byte_width):
                byte = sline[i:i + byte_width]
                if byte == bytea:
                    ia = i // byte_width
                if byte == byteb:
                    ib = i // byte_width
            if ia >= 0 and ib >= 0:
                maska = 0b1111 << (tl - (ia + 1) * byte_width)
                maskb = 0b1111 << (tl - (ib + 1) * byte_width)
                if ia < ib:
                    y = line >> (abs(ib - ia) * byte_width) & maskb
                    z = line << (abs(ia - ib) * byte_width) & maska
                else:
                    y = line << (abs(ib - ia) * byte_width) & maskb
                    z = line >> (abs(ia - ib) * byte_width) & maska
                line = line & ~(maska | maskb) | (y | z)
            else:
                # print(ia, ib)
                print('error: couldn\'t find the required byte')
        # print(move, translate(line))
    return translate(line)

In [262]:
def catch_a_groove4(line):
    for move in moves:
        if move['type'] == 's':
            l = move['value'] * byte_width
            line = ((line << (tl - l)) ^ (line >> l)) & mask
        if move['type'] == 'x':
            ia, ib = move['value']
            maska = 0b1111 << (tl - (ia + 1) * byte_width)
            maskb = 0b1111 << (tl - (ib + 1) * byte_width)
            if ia < ib:
                a = line >> (abs(ib - ia) * byte_width) & maskb
                b = line << (abs(ia - ib) * byte_width) & maska
            else:
                a = line << (abs(ib - ia) * byte_width) & maskb
                b = line >> (abs(ia - ib) * byte_width) & maska
            
            line = line & ~(maska | maskb) | (a | b)
        if move['type'] == 'p':
            a, b = move['value']
            # line = list(translate(line))
            # ia = line.index(a)
            # ib = line.index(b)
            # line[ia] = b
            # line[ib] = a
            # line = int(to_bytes(line), 2)
            loca = ref[a]
            locb = ref[b]
            bytea = f'{loca:04b}'
            byteb = f'{locb:04b}'
            sline = f'{line:064b}'
            ia = -1
            ib = -1
            for i in range(0, tl, byte_width):
                byte = sline[i:i + byte_width]
                if byte == bytea:
                    ia = i // byte_width
                if byte == byteb:
                    ib = i // byte_width
            if ia >= 0 and ib >= 0:
                maska = 0b1111 << (tl - (ia + 1) * byte_width)
                maskb = 0b1111 << (tl - (ib + 1) * byte_width)
                if ia < ib:
                    y = line >> (abs(ib - ia) * byte_width) & maskb
                    z = line << (abs(ia - ib) * byte_width) & maska
                else:
                    y = line << (abs(ib - ia) * byte_width) & maskb
                    z = line >> (abs(ia - ib) * byte_width) & maska
                line = line & ~(maska | maskb) | (y | z)
            else:
                # print(ia, ib)
                print('error: couldn\'t find the required byte')
        # print(move, translate(line))
    return translate(line)

In [255]:
def to_bytes(line):
    result = ''
    for char in line:
        b = ref[char]
        result += '{0:04b}'.format(b)

    return result

In [256]:
to_bytes(['a', 'b', 'c'])

'000000010010'

In [263]:
%%prun
catch_a_groove4(line2)

         14109 function calls in 0.139 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
       10    0.118    0.012    0.121    0.012 socket.py:623(send)
        1    0.007    0.007    0.024    0.024 history.py:1050(_writeout_input_cache)
        2    0.006    0.003    0.006    0.003 {method '__exit__' of 'sqlite3.Connection' objects}
    13822    0.006    0.000    0.006    0.000 {built-in method builtins.abs}
        1    0.002    0.002    0.002    0.002 3319956737.py:1(catch_a_groove4)
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}
        1    0.000    0.000    0.036    0.036 <string>:1(<module>)
       10    0.000    0.000    0.000    0.000 enum.py:1583(__or__)
       30    0.000    0.000    0.000    0.000 enum.py:1576(_get_value)
       64    0.000    0.000    0.000    0.000 {built-in method builtins.isinstance}
        1    0.000    0.000    0.000    0.000 threading.py:32

In [71]:
%%timeit
catch_a_groove3(line)

4.28 ms ± 469 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [241]:
def translate(n):
    s = '{0:064b}'.format(n)
    result = []
    for i in range(0, 64, 4):
        byte = s[i:i+4]
        j = int(byte, 2)
        char = 'abcdefghijklmnop'[j]
        result.append(char)
    return ''.join(result)

In [299]:
a = 'a'
b = 'b'
x = 0b00011011
mask = 0b11111111
tl = 8
byte_width = 2

In [303]:
# try switching a with b to get 01001011
ref = {y: i for i, y in enumerate(list('abcd'))}
loca = ref[a]
locb = ref[b]
bytea = f'{loca:02b}'
byteb = f'{locb:02b}'
sx = f'{x:08b}'

In [308]:
ia = -1
ib = -1
for i in range(0, tl, byte_width):
    byte = sx[i:i+byte_width]
    if byte == bytea:
        ia = i // byte_width
    if byte == byteb:
        ib = i // byte_width
if ia >= 0 and ib >= 0:
    maska = 0b11 << (tl - (ia + 1) * 2)
    maskb = 0b11 << (tl - (ib + 1) * 2)
    if ia < ib:
        y = x >> (abs(ib - ia) * 2) & maskb
        z = x << (abs(ia - ib) * 2) & maska
    else:
        y = x << (abs(ib - ia) * 2) & maskb
        z = x >> (abs(ia - ib) * 2) & maska
    result = x & ~(maska | maskb) | (y | z)
    print(f'{result:08b}')
else:
    print('error: couldn\'t find the required byte')

01001011


In [252]:
# 00011011
# 00000100
# 00100000
bin(x & ~(maska | maskb) | (a | b))

'0b11110'

In [175]:
0xffffffffffffffff

18446744073709551615

In [171]:
0b1111111111111111111111111111111111111111111111111111111111111111

18446744073709551615

In [276]:
line2

81985529216486895

In [226]:
catch_a_groove3(line)

'doeaimlbnpjchfkg'

In [86]:
line = list('abcdefghijklmnop')
# start = datetime.datetime.now()
for x in range(1_000_000_000 % 30):
    line = catch_a_groove3(line)
    # if line == list('abcdefghijklmnop'):
    #     print(x)
    #     break
''.join(line)

'agndefjhibklmocp'

In [273]:
def catch_a_groove5(line):
    for move in moves:
        if move['type'] == 's':
            l = move['value']
            for char in line:
                line[char] = (line[char] + l) % 16
        elif move['type'] == 'x':
            ia, ib = move['value']
            a = ''
            b = ''
            for char, index in line.items():
                if index == ia:
                    a = char
                elif index == ib:
                    b = char
            line[a] = ib
            line[b] = ia
        elif move['type'] == 'p':
            a, b = move['value']
            ia = line[a]
            ib = line[b]
            line[a] = ib
            line[b] = ia

    return line

In [275]:
%%timeit
catch_a_groove5(ref)

16.3 ms ± 1.28 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [276]:
counter = {'s': 0, 'x': 0, 'p': 0}
for move in moves:
    counter[move['type']] += 1

In [277]:
counter

{'s': 3089, 'x': 4681, 'p': 2230}

In [72]:
math.factorial(16)

20922789888000